# 🔬 Explainable TinyML for PPG-Based Oxygen Risk Monitoring
## Complete ML Workflow — Visual Pipeline

This notebook provides a **comprehensive, richly visual walkthrough** of the entire machine learning pipeline for the final-year capstone project:

> *Design and Implementation of a Lightweight Explainable TinyML Framework for Real-Time Oxygen Monitoring on Edge Devices*

---

### Table of Contents

| Section | Description |
| :--- | :--- |
| **§1** | Setup & Configuration |
| **§2** | Dataset Overview & Class Distribution |
| **§3** | Feature Engineering & Exploration |
| **§4** | Feature Correlation & Redundancy Analysis |
| **§5** | Baseline Model — Logistic Regression |
| **§6** | Candidate Model — TinyKAN (B-Spline KAN) |
| **§7** | Model Comparison & Cross-Validation |
| **§8** | Decision Threshold Optimization |
| **§9** | Hyperparameter Sweep Analysis |
| **§10** | Global Explainability — Feature Importance |
| **§11** | Local Explainability — Per-Window Attribution |
| **§12** | Explanation Quality Rubric Scoring |
| **§13** | TinyML Deployment Summary |

---

## §1 — Setup & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, classification_report
)
from scipy import signal, stats

# ── Project Paths ──────────────────────────────────────────────────────────────
PROJECT_ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"

# ── Visual Theme ───────────────────────────────────────────────────────────────
# Custom dark theme with clinical palette
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'savefig.facecolor': '#0d1117',
})

# Color palette
C = {
    'risk':     '#f85149',   # Red — low-oxygen risk
    'normal':   '#58a6ff',   # Blue — normal oxygen
    'accent':   '#3fb950',   # Green — accent / pass
    'warn':     '#d29922',   # Amber — warning
    'purple':   '#bc8cff',   # Purple — secondary accent  
    'cyan':     '#39d2c0',   # Cyan — tertiary
    'text':     '#c9d1d9',   # Light text
    'dim':      '#8b949e',   # Dim text
    'bg':       '#161b22',   # Background
    'grid':     '#21262d',   # Grid lines
    'border':   '#30363d',   # Border
}

# Feature metadata
FEATURE_NAMES = (
    "mean", "std", "minimum", "maximum", "value_range", "rms",
    "skewness", "kurtosis", "energy", "derivative_std",
    "dominant_frequency_hz", "bandpower_0_5_2_hz",
    "bandpower_2_5_hz", "bandpower_ratio_0_5_2_to_2_5",
)

FEATURE_LABELS = {
    "mean": "Mean Amplitude",
    "std": "PPG Variability",
    "minimum": "Min Amplitude",
    "maximum": "Max Amplitude",
    "value_range": "Amplitude Range",
    "rms": "Signal Power (RMS)",
    "skewness": "Skewness",
    "kurtosis": "Kurtosis",
    "energy": "Spectral Energy",
    "derivative_std": "Slope Variability",
    "dominant_frequency_hz": "Dominant Freq.",
    "bandpower_0_5_2_hz": "Bandpower (0.5–2 Hz)",
    "bandpower_2_5_hz": "Bandpower (2–5 Hz)",
    "bandpower_ratio_0_5_2_to_2_5": "Bandpower Ratio",
}

FEATURE_CATEGORIES = {
    "Amplitude": ["mean", "std", "minimum", "maximum", "value_range", "rms"],
    "Morphology": ["skewness", "kurtosis"],
    "Spectral": ["energy", "dominant_frequency_hz", "bandpower_0_5_2_hz", "bandpower_2_5_hz", "bandpower_ratio_0_5_2_to_2_5"],
    "Derivative": ["derivative_std"],
}

print("✅ Setup complete — dark theme applied")
print(f"📂 Project root: {PROJECT_ROOT}")

---
## §2 — Dataset Overview & Class Distribution

We use the **OpenOximetry** dataset — a large public repository of pulse oximetry recordings from the PhysioNet archive.

Each sample is a **10-second sliding window** of raw PPG waveform data (50 Hz, 500 samples/window).
Labels are derived from concurrent reference SpO₂ readings:
- **Label 0 (Normal)**: Median SpO₂ ≥ 90%
- **Label 1 (Low-Oxygen Risk)**: Median SpO₂ < 90%

In [ ]:
# ── Load main feature dataset ──────────────────────────────────────────────────
feat_csv = OUTPUTS / "openox_canonical_window_features.csv"
df = pd.read_csv(feat_csv)
df["label"] = df["label"].astype(int)
df["recording_id"] = df["recording_id"].astype(str)

n_total = len(df)
n_pos = int(df["label"].sum())
n_neg = n_total - n_pos
n_recordings = df["recording_id"].nunique()

print(f"📊 Dataset Summary")
print(f"   Total windows:      {n_total:,}")
print(f"   Normal (label=0):   {n_neg:,}  ({n_neg/n_total:.1%})")
print(f"   Risk   (label=1):   {n_pos:,}  ({n_pos/n_total:.1%})")
print(f"   Unique recordings:  {n_recordings}")
print(f"   Features:           {len(FEATURE_NAMES)}")

In [ ]:
# ── Class Distribution Visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Bar chart ──
ax = axes[0]
bars = ax.bar(["Normal\n(SpO₂ ≥ 90%)", "Low-O₂ Risk\n(SpO₂ < 90%)"],
              [n_neg, n_pos],
              color=[C['normal'], C['risk']],
              edgecolor='none', width=0.55, zorder=3)
for bar, count in zip(bars, [n_neg, n_pos]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
            f"{count:,}\n({count/n_total:.1%})",
            ha='center', va='bottom', fontsize=11, fontweight='bold', color=C['text'])
ax.set_ylabel("Number of Windows", fontweight='bold')
ax.set_title("Class Distribution", fontweight='bold', pad=12)
ax.set_ylim(0, max(n_neg, n_pos) * 1.22)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 2: Donut chart ──
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    [n_neg, n_pos], labels=['Normal', 'Risk'],
    colors=[C['normal'], C['risk']],
    autopct='%1.1f%%', startangle=90, pctdistance=0.78,
    wedgeprops=dict(width=0.4, edgecolor=C['bg'], linewidth=2),
    textprops=dict(color=C['text'], fontsize=11, fontweight='bold'))
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax.set_title("Class Proportion", fontweight='bold', pad=12)
centre = plt.Circle((0,0), 0.55, fc=C['bg'])
ax.add_artist(centre)
ax.text(0, 0, f"{n_total:,}\nwindows", ha='center', va='center',
        fontsize=12, fontweight='bold', color=C['text'])

# ── Panel 3: Per-recording class ratio ──
ax = axes[2]
rec_stats = df.groupby("recording_id")["label"].agg(['sum', 'count'])
rec_stats['risk_ratio'] = rec_stats['sum'] / rec_stats['count']
rec_stats_sorted = rec_stats.sort_values('risk_ratio')
ax.barh(range(len(rec_stats_sorted)), rec_stats_sorted['risk_ratio'],
        color=[C['risk'] if r > 0.5 else C['normal'] for r in rec_stats_sorted['risk_ratio']],
        height=1.0, edgecolor='none', alpha=0.8)
ax.axvline(0.5, color=C['warn'], linewidth=1.5, linestyle='--', alpha=0.7, label='50% boundary')
ax.set_xlabel("Risk Window Fraction", fontweight='bold')
ax.set_ylabel("Recording (sorted)", fontweight='bold')
ax.set_title(f"Risk Ratio per Recording (n={n_recordings})", fontweight='bold', pad=12)
ax.set_yticks([])
ax.legend(fontsize=9, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle("OpenOximetry Dataset — 52K PPG Windows from 162 Subjects",
             fontsize=16, fontweight='bold', color=C['text'], y=1.02)
plt.tight_layout()
plt.show()

---
## §3 — Feature Engineering & Exploration

The system extracts **14 physiological features** from each 10-second PPG window:

| Category | Features | Count |
| :--- | :--- | :---: |
| **Amplitude** | mean, std, min, max, range, RMS | 6 |
| **Morphology** | skewness, kurtosis | 2 |
| **Spectral** | energy, dominant freq, bandpower (0.5–2 Hz), bandpower (2–5 Hz), bandpower ratio | 5 |
| **Derivative** | derivative std | 1 |

In [ ]:
# ── Feature Distribution Violin Plots ────────────────────────────────────────
X = df[list(FEATURE_NAMES)].copy()
y = df["label"].values

# Standardize for comparable visualization
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURE_NAMES)
X_scaled["label"] = y

fig, axes = plt.subplots(2, 7, figsize=(22, 9), sharey=True)
axes = axes.ravel()

for i, feat in enumerate(FEATURE_NAMES):
    ax = axes[i]
    data_neg = X_scaled.loc[X_scaled['label'] == 0, feat].values
    data_pos = X_scaled.loc[X_scaled['label'] == 1, feat].values
    
    # Subsample for speed
    np.random.seed(42)
    data_neg_s = np.random.choice(data_neg, min(2000, len(data_neg)), replace=False)
    data_pos_s = np.random.choice(data_pos, min(2000, len(data_pos)), replace=False)
    
    parts = ax.violinplot([data_neg_s, data_pos_s], positions=[0, 1],
                          showmeans=True, showmedians=True, showextrema=False)
    for pc, color in zip(parts['bodies'], [C['normal'], C['risk']]):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts['cmeans'].set_color(C['warn'])
    parts['cmedians'].set_color(C['text'])
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['N', 'R'], fontsize=9)
    ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=9, fontweight='bold', pad=4)
    ax.grid(axis='y', alpha=0.2)
    ax.set_ylim(-4, 4)

fig.suptitle("Feature Distributions by Class (Standardized)",
             fontsize=16, fontweight='bold', color=C['text'], y=1.01)

# Legend
legend_elements = [
    mpatches.Patch(facecolor=C['normal'], alpha=0.7, label='Normal (N)'),
    mpatches.Patch(facecolor=C['risk'], alpha=0.7, label='Risk (R)'),
]
fig.legend(handles=legend_elements, loc='upper right', fontsize=10,
           bbox_to_anchor=(0.98, 0.98), frameon=True,
           facecolor=C['bg'], edgecolor=C['border'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Category Radar Chart ─────────────────────────────────────────────
# Show mean standardized feature values per class as a radar / spider chart

mean_normal = X_scaled[X_scaled['label'] == 0][list(FEATURE_NAMES)].mean()
mean_risk = X_scaled[X_scaled['label'] == 1][list(FEATURE_NAMES)].mean()

labels = [FEATURE_LABELS.get(f, f) for f in FEATURE_NAMES]
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

vals_normal = mean_normal.values.tolist() + [mean_normal.values[0]]
vals_risk = mean_risk.values.tolist() + [mean_risk.values[0]]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_facecolor(C['bg'])

ax.plot(angles, vals_normal, 'o-', linewidth=2, color=C['normal'], label='Normal', markersize=5)
ax.fill(angles, vals_normal, alpha=0.15, color=C['normal'])
ax.plot(angles, vals_risk, 'o-', linewidth=2, color=C['risk'], label='Risk', markersize=5)
ax.fill(angles, vals_risk, alpha=0.15, color=C['risk'])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=8, fontweight='bold')
ax.set_title("Mean Feature Profile by Class (Standardized)",
             fontsize=14, fontweight='bold', pad=25, color=C['text'])
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1.1), fontsize=11,
          frameon=True, facecolor=C['bg'], edgecolor=C['border'])
ax.grid(color=C['grid'], alpha=0.5)
ax.tick_params(colors=C['dim'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Histogram Grid: Raw Feature Distributions ────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(22, 11))
axes_flat = axes.ravel()

for i, feat in enumerate(FEATURE_NAMES):
    ax = axes_flat[i]
    vals_n = df.loc[df['label'] == 0, feat].values
    vals_r = df.loc[df['label'] == 1, feat].values
    
    # Clip to ±3σ for visualization
    combined = df[feat].values
    mu, sigma = combined.mean(), combined.std()
    lo, hi = mu - 3*sigma, mu + 3*sigma
    
    bins = np.linspace(lo, hi, 60)
    ax.hist(vals_n, bins=bins, alpha=0.6, color=C['normal'], label='Normal', density=True, edgecolor='none')
    ax.hist(vals_r, bins=bins, alpha=0.6, color=C['risk'], label='Risk', density=True, edgecolor='none')
    ax.set_title(FEATURE_LABELS.get(feat, feat), fontsize=10, fontweight='bold', pad=4)
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

# Hide unused subplot
axes_flat[-1].set_visible(False)

axes_flat[0].legend(fontsize=9, loc='upper right')
fig.suptitle("Raw Feature Distributions — Normal vs. Low-Oxygen Risk",
             fontsize=16, fontweight='bold', color=C['text'], y=1.01)
plt.tight_layout()
plt.show()

---
## §4 — Feature Correlation & Redundancy Analysis

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
corr = df[list(FEATURE_NAMES)].corr()
labels_short = [FEATURE_LABELS.get(f, f) for f in FEATURE_NAMES]

# Custom diverging colormap: blue → dark → red
cmap = LinearSegmentedColormap.from_list(
    'custom_diverging',
    [(0.0, C['normal']), (0.5, C['bg']), (1.0, C['risk'])]
)

fig, ax = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
im = ax.imshow(corr.where(~mask).values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(labels_short)))
ax.set_yticks(range(len(labels_short)))
ax.set_xticklabels(labels_short, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels_short, fontsize=9)

# Annotate cells
for i in range(len(corr)):
    for j in range(len(corr)):
        if not mask[i, j]:
            val = corr.iloc[i, j]
            text_color = C['text'] if abs(val) > 0.4 else C['dim']
            ax.text(j, i, f"{val:.2f}", ha='center', va='center',
                    fontsize=7, color=text_color, fontweight='bold' if abs(val) > 0.7 else 'normal')

cbar = fig.colorbar(im, ax=ax, shrink=0.75, label='Pearson r')
cbar.ax.yaxis.label.set_color(C['text'])
cbar.ax.tick_params(colors=C['dim'])

ax.set_title("Feature Correlation Matrix (Lower Triangle)",
             fontsize=14, fontweight='bold', pad=15, color=C['text'])
plt.tight_layout()
plt.show()

# Report highly correlated pairs
print("\n🔗 Highly correlated feature pairs (|r| > 0.8):")
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        r = corr.iloc[i, j]
        if abs(r) > 0.8:
            print(f"   {FEATURE_LABELS[FEATURE_NAMES[i]]:25s} ↔ {FEATURE_LABELS[FEATURE_NAMES[j]]:25s}  r = {r:+.3f}")

In [ ]:
# ── Clustermap for visual grouping ───────────────────────────────────────────
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

# Distance = 1 - |correlation|
dist = 1 - np.abs(corr.values)
np.fill_diagonal(dist, 0)
condensed = squareform(dist)
Z = linkage(condensed, method='ward')

fig, (ax_dend, ax_heat) = plt.subplots(1, 2, figsize=(16, 7),
                                        gridspec_kw={'width_ratios': [1, 2.5]})

# Dendrogram
dn = dendrogram(Z, labels=labels_short, orientation='left', ax=ax_dend,
                leaf_font_size=9, color_threshold=0.6,
                above_threshold_color=C['dim'])
ax_dend.set_title("Feature Hierarchy", fontweight='bold', pad=10)
ax_dend.spines['top'].set_visible(False)
ax_dend.spines['right'].set_visible(False)
ax_dend.spines['bottom'].set_visible(False)

# Reordered heatmap
order = dn['leaves']
corr_ordered = corr.iloc[order, order]
im = ax_heat.imshow(corr_ordered.values, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax_heat.set_xticks(range(len(order)))
ax_heat.set_yticks(range(len(order)))
ax_heat.set_xticklabels([labels_short[i] for i in order], rotation=45, ha='right', fontsize=9)
ax_heat.set_yticklabels([labels_short[i] for i in order], fontsize=9)
ax_heat.set_title("Hierarchically Clustered Correlation", fontweight='bold', pad=10)
fig.colorbar(im, ax=ax_heat, shrink=0.6, label='Pearson r')

plt.tight_layout()
plt.show()

---
## §5 — Baseline Model: Logistic Regression

The **Logistic Regression (L2)** baseline uses `GroupKFold` (5-fold, split by recording ID) to prevent data leakage.

Local explanation for LR is trivially: $C_i = w_i \times x_{i,\text{scaled}}$ — zero extra forward passes.

In [ ]:
# ── Load pre-computed results ────────────────────────────────────────────────
lr_summary = pd.read_csv(OUTPUTS / "openox_logreg_baseline_summary.csv")
lr_diag = pd.read_csv(OUTPUTS / "openox_logreg_split_diagnostics.csv")
lr_primary = lr_summary[lr_summary["label_variant"] == "label"].iloc[0]
lr_folds = lr_diag[lr_diag["label_variant"] == "label"].copy()

# ── Confusion Matrix ──
cm_lr = np.array([
    [int(lr_primary["pooled_true_negative"]), int(lr_primary["pooled_false_positive"])],
    [int(lr_primary["pooled_false_negative"]), int(lr_primary["pooled_true_positive"])],
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Panel 1: Confusion Matrix ──
ax = axes[0]
cm_cmap = LinearSegmentedColormap.from_list('cm_blue', ['#0d1117', '#1f6feb', '#58a6ff'])
im = ax.imshow(cm_lr, cmap=cm_cmap, aspect='auto')
for i in range(2):
    for j in range(2):
        color = '#ffffff' if cm_lr[i,j] > cm_lr.max()/2 else C['text']
        ax.text(j, i, f"{cm_lr[i,j]:,}", ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)

labels_cm = ['TN', 'FP', 'FN', 'TP']
for i in range(2):
    for j in range(2):
        label = labels_cm[i*2 + j]
        ax.text(j, i + 0.32, label, ha='center', va='center',
                fontsize=9, color=C['dim'], style='italic')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Negative', 'Predicted Positive'], fontsize=10)
ax.set_yticklabels(['Actual Negative', 'Actual Positive'], fontsize=10)
ax.set_title('Logistic Regression — Pooled Confusion Matrix', fontweight='bold', pad=12)
ax.spines[:].set_visible(False)

# ── Panel 2: Metrics bar ──
ax = axes[1]
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_vals = [
    lr_primary['pooled_accuracy'],
    lr_primary['pooled_precision'],
    lr_primary['pooled_recall'],
    lr_primary['pooled_f1']
]
colors_bar = [C['purple'], C['cyan'], C['risk'], C['accent']]

bars = ax.barh(metrics_names[::-1], metrics_vals[::-1],
               color=colors_bar[::-1], height=0.55, edgecolor='none', zorder=3)
for bar, val in zip(bars, metrics_vals[::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", ha='left', va='center', fontsize=11, fontweight='bold', color=C['text'])
ax.set_xlim(0, 1.05)
ax.set_title('Pooled Classification Metrics (LR)', fontweight='bold', pad=12)
ax.grid(axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\n📋 Logistic Regression Baseline (GroupKFold k=5):")
print(f"   Accuracy:  {lr_primary['pooled_accuracy']:.4f}")
print(f"   Precision: {lr_primary['pooled_precision']:.4f}")
print(f"   Recall:    {lr_primary['pooled_recall']:.4f}")
print(f"   F1-Score:  {lr_primary['pooled_f1']:.4f}")

In [ ]:
# ── ROC + Precision-Recall Curves (recomputed from data) ─────────────────────
# Reproduce the LR model to get per-sample probabilities for ROC/PR curves

X_all = df[list(FEATURE_NAMES)].values.astype(np.float64)
y_all = df["label"].values
groups = df["recording_id"].values

splitter = GroupKFold(n_splits=5)
all_probs = np.zeros(len(y_all))

for train_idx, test_idx in splitter.split(X_all, y_all, groups=groups):
    scaler_fold = StandardScaler()
    X_tr = scaler_fold.fit_transform(X_all[train_idx])
    X_te = scaler_fold.transform(X_all[test_idx])
    
    model_fold = LogisticRegression(class_weight='balanced', max_iter=1000,
                                     solver='liblinear', random_state=42)
    model_fold.fit(X_tr, y_all[train_idx])
    all_probs[test_idx] = model_fold.predict_proba(X_te)[:, 1]

# ROC
fpr, tpr, roc_thresholds = roc_curve(y_all, all_probs)
roc_auc = auc(fpr, tpr)

# PR
prec_curve, rec_curve, pr_thresholds = precision_recall_curve(y_all, all_probs)
ap = average_precision_score(y_all, all_probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ── ROC Curve ──
ax1.plot(fpr, tpr, color=C['normal'], linewidth=2.5, label=f'LR (AUC = {roc_auc:.3f})')
ax1.fill_between(fpr, tpr, alpha=0.12, color=C['normal'])
ax1.plot([0, 1], [0, 1], '--', color=C['dim'], linewidth=1, label='Random baseline')
ax1.set_xlabel('False Positive Rate', fontweight='bold')
ax1.set_ylabel('True Positive Rate', fontweight='bold')
ax1.set_title('ROC Curve — Logistic Regression', fontweight='bold', pad=12)
ax1.legend(fontsize=10, loc='lower right')
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(-0.02, 1.02)
ax1.grid(alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Precision-Recall Curve ──
ax2.plot(rec_curve, prec_curve, color=C['risk'], linewidth=2.5, label=f'LR (AP = {ap:.3f})')
ax2.fill_between(rec_curve, prec_curve, alpha=0.12, color=C['risk'])
ax2.axhline(n_pos/n_total, color=C['dim'], linewidth=1, linestyle='--', label=f'Random ({n_pos/n_total:.2f})')
ax2.set_xlabel('Recall', fontweight='bold')
ax2.set_ylabel('Precision', fontweight='bold')
ax2.set_title('Precision-Recall Curve — Logistic Regression', fontweight='bold', pad=12)
ax2.legend(fontsize=10, loc='upper right')
ax2.set_xlim(-0.02, 1.02)
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"   ROC AUC:          {roc_auc:.4f}")
print(f"   Average Precision: {ap:.4f}")

---
## §6 — Candidate Model: TinyKAN (B-Spline KAN)

The **TinyKAN** classifier uses a Kolmogorov-Arnold Network with B-spline edge activations:

- Architecture: `(14, 12, 1)` — 14 input features → 12 hidden → 1 output
- B-splines: 3rd degree, 8 knots, range [-3, 3]
- Total parameters: **297**
- Decision threshold: **0.30** (optimized for recall)

In [ ]:
# ── Load TinyKAN results ─────────────────────────────────────────────────────
tk_summary = pd.read_csv(OUTPUTS / "openox_tinykan_summary.csv")
tk_diag = pd.read_csv(OUTPUTS / "openox_tinykan_split_diagnostics.csv")
tk = tk_summary.iloc[0]

# ── TinyKAN Confusion Matrix ──
cm_tk = np.array([
    [int(tk["pooled_true_negative"]), int(tk["pooled_false_positive"])],
    [int(tk["pooled_false_negative"]), int(tk["pooled_true_positive"])],
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ── Panel 1: TinyKAN Confusion Matrix ──
ax = axes[0]
cm_cmap_r = LinearSegmentedColormap.from_list('cm_red', ['#0d1117', '#da3633', '#f85149'])
im = ax.imshow(cm_tk, cmap=cm_cmap_r, aspect='auto')
for i in range(2):
    for j in range(2):
        color = '#ffffff' if cm_tk[i,j] > cm_tk.max()/2 else C['text']
        ax.text(j, i, f"{cm_tk[i,j]:,}", ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)

labels_cm = ['TN', 'FP', 'FN', 'TP']
for i in range(2):
    for j in range(2):
        label = labels_cm[i*2 + j]
        ax.text(j, i + 0.32, label, ha='center', va='center',
                fontsize=9, color=C['dim'], style='italic')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted Negative', 'Predicted Positive'], fontsize=10)
ax.set_yticklabels(['Actual Negative', 'Actual Positive'], fontsize=10)
ax.set_title('TinyKAN — Pooled Confusion Matrix', fontweight='bold', pad=12)
ax.spines[:].set_visible(False)

# ── Panel 2: Metrics ──
ax = axes[1]
metrics_vals_tk = [
    tk['pooled_accuracy'],
    tk['pooled_precision'],
    tk['pooled_recall'],
    tk['pooled_f1']
]
bars = ax.barh(metrics_names[::-1], metrics_vals_tk[::-1],
               color=colors_bar[::-1], height=0.55, edgecolor='none', zorder=3)
for bar, val in zip(bars, metrics_vals_tk[::-1]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", ha='left', va='center', fontsize=11, fontweight='bold', color=C['text'])
ax.set_xlim(0, 1.05)
ax.set_title('Pooled Classification Metrics (TinyKAN)', fontweight='bold', pad=12)
ax.grid(axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f"\n📋 TinyKAN (14→12→1) with B-Spline KAN:")
print(f"   Parameters: {int(tk['parameter_count'])}")
print(f"   Accuracy:   {tk['pooled_accuracy']:.4f}")
print(f"   Precision:  {tk['pooled_precision']:.4f}")
print(f"   Recall:     {tk['pooled_recall']:.4f}  ← optimised for safety")
print(f"   F1-Score:   {tk['pooled_f1']:.4f}")

---
## §7 — Model Comparison & Cross-Validation

Head-to-head comparison of the two models evaluated on identical GroupKFold splits.

In [ ]:
# ── Side-by-Side Model Comparison ────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

# ── Panel 1: Grouped Bar Comparison ──
ax = fig.add_subplot(gs[0, 0:2])
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
lr_vals = [lr_primary['pooled_accuracy'], lr_primary['pooled_precision'],
           lr_primary['pooled_recall'], lr_primary['pooled_f1']]
tk_vals_cmp = [tk['pooled_accuracy'], tk['pooled_precision'],
           tk['pooled_recall'], tk['pooled_f1']]

x = np.arange(len(metrics))
w = 0.32
bars1 = ax.bar(x - w/2, lr_vals, w, label='Logistic Regression', color=C['normal'],
               edgecolor='none', zorder=3)
bars2 = ax.bar(x + w/2, tk_vals_cmp, w, label='TinyKAN (14→12→1)', color=C['risk'],
               edgecolor='none', zorder=3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                f"{h:.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold', color=C['text'])

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score', fontweight='bold')
ax.set_ylim(0, 1.08)
ax.set_title('Pooled Metric Comparison — LR vs TinyKAN', fontweight='bold', pad=12)
ax.legend(fontsize=10, loc='upper left', frameon=True, facecolor=C['bg'], edgecolor=C['border'])
ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 2: Model Summary Card ──
ax = fig.add_subplot(gs[0, 2])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

card_text = (
    "MODEL COMPARISON CARD\n"
    "━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    f"Logistic Regression\n"
    f"  Parameters:  15\n"
    f"  Recall:      {lr_primary['pooled_recall']:.1%}\n"
    f"  F1-Score:    {lr_primary['pooled_f1']:.3f}\n"
    f"  XAI Cost:    O(N) — zero passes\n\n"
    f"TinyKAN (B-Spline)\n"
    f"  Parameters:  297\n"
    f"  Recall:      {tk['pooled_recall']:.1%}  ↑\n"
    f"  F1-Score:    {tk['pooled_f1']:.3f}  ↑\n"
    f"  XAI Cost:    15 fwd passes\n\n"
    f"━━━━━━━━━━━━━━━━━━━━━━━\n"
    f"Dataset: 52,090 windows\n"
    f"CV: GroupKFold (k=5)\n"
    f"Subjects: 162"
)

ax.text(0.5, 0.5, card_text, ha='center', va='center', fontsize=9,
        fontfamily='monospace', color=C['text'],
        bbox=dict(boxstyle='round,pad=0.8', facecolor=C['bg'],
                  edgecolor=C['border'], linewidth=1.5))

# ── Panel 3-6: Fold-wise Metrics ──
fold_metrics = ['accuracy', 'precision', 'recall', 'f1']
fold_titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for i, (metric, title) in enumerate(zip(fold_metrics, fold_titles)):
    ax = fig.add_subplot(gs[1, i]) if i < 3 else None
    if ax is None:
        break
    
    lr_fold_vals = lr_folds[metric].values
    tk_fold_vals = tk_diag[metric].values
    folds = range(1, 6)
    
    ax.plot(folds, lr_fold_vals, 'o-', color=C['normal'], linewidth=2.5,
            markersize=8, label='LR', markeredgecolor='white', markeredgewidth=1)
    ax.plot(folds, tk_fold_vals, 's--', color=C['risk'], linewidth=2.5,
            markersize=8, label='TinyKAN', markeredgecolor='white', markeredgewidth=1)
    
    # Fill between
    ax.fill_between(folds, lr_fold_vals, tk_fold_vals, alpha=0.08, color=C['warn'])
    
    ax.set_xticks(list(folds))
    ax.set_xlabel('Fold', fontweight='bold')
    ax.set_title(title, fontweight='bold', pad=8)
    ax.grid(axis='y', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if i == 0:
        ax.set_ylabel('Score', fontweight='bold')
    if i == 2:
        ax.legend(fontsize=9, loc='lower left', frameon=True,
                  facecolor=C['bg'], edgecolor=C['border'])

fig.suptitle("Model Comparison — Logistic Regression vs TinyKAN",
             fontsize=17, fontweight='bold', color=C['text'], y=1.02)
plt.show()

In [ ]:
# ── Cross-Validation Stability Analysis ──────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for i, (metric, title) in enumerate(zip(fold_metrics, fold_titles)):
    ax = axes[i]
    
    lr_vals_f = lr_folds[metric].values
    tk_vals_f = tk_diag[metric].values
    
    bp = ax.boxplot([lr_vals_f, tk_vals_f], labels=['LR', 'TinyKAN'],
                    patch_artist=True, widths=0.45,
                    boxprops=dict(linewidth=1.5),
                    whiskerprops=dict(color=C['dim'], linewidth=1.5),
                    capprops=dict(color=C['dim'], linewidth=1.5),
                    medianprops=dict(color=C['warn'], linewidth=2),
                    flierprops=dict(marker='o', markerfacecolor=C['dim'], markersize=5))
    
    bp['boxes'][0].set_facecolor(C['normal'])
    bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor(C['risk'])
    bp['boxes'][1].set_alpha(0.6)
    
    # Overlay individual fold points
    ax.scatter([1]*5, lr_vals_f, color=C['normal'], s=40, zorder=5, edgecolors='white', linewidth=0.8)
    ax.scatter([2]*5, tk_vals_f, color=C['risk'], s=40, zorder=5, edgecolors='white', linewidth=0.8)
    
    ax.set_title(title, fontweight='bold', pad=8)
    ax.grid(axis='y', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Annotate mean ± std
    ax.text(1, lr_vals_f.max() + 0.003, f"μ={lr_vals_f.mean():.3f}",
            ha='center', fontsize=8, color=C['normal'], fontweight='bold')
    ax.text(2, tk_vals_f.max() + 0.003, f"μ={tk_vals_f.mean():.3f}",
            ha='center', fontsize=8, color=C['risk'], fontweight='bold')

fig.suptitle("Cross-Validation Stability — 5-Fold Distribution",
             fontsize=15, fontweight='bold', color=C['text'], y=1.04)
plt.tight_layout()
plt.show()

---
## §8 — Decision Threshold Optimisation

For clinical safety, we optimise the decision threshold to **maximise recall** (sensitivity to low-oxygen events) while maintaining acceptable precision.

In [ ]:
# ── Threshold Sweep (LR) ─────────────────────────────────────────────────────
thresh_df = pd.read_csv(OUTPUTS / "openox_logreg_threshold_sweep.csv")

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(thresh_df['threshold'], thresh_df['pooled_accuracy'], 'o-',
        color=C['purple'], linewidth=2.5, markersize=8, label='Accuracy',
        markeredgecolor='white', markeredgewidth=1)
ax.plot(thresh_df['threshold'], thresh_df['pooled_precision'], 's-',
        color=C['cyan'], linewidth=2.5, markersize=8, label='Precision',
        markeredgecolor='white', markeredgewidth=1)
ax.plot(thresh_df['threshold'], thresh_df['pooled_recall'], '^-',
        color=C['risk'], linewidth=2.5, markersize=8, label='Recall',
        markeredgecolor='white', markeredgewidth=1)
ax.plot(thresh_df['threshold'], thresh_df['pooled_f1'], 'D-',
        color=C['accent'], linewidth=2.5, markersize=8, label='F1-Score',
        markeredgecolor='white', markeredgewidth=1)

# Highlight the 0.5 threshold (LR default)
ax.axvline(0.5, color=C['normal'], linewidth=2, linestyle='--', alpha=0.7, label='Default (0.50)')

# Annotate best F1 threshold
best_idx = thresh_df['pooled_f1'].idxmax()
best_thresh = thresh_df.loc[best_idx, 'threshold']
best_f1 = thresh_df.loc[best_idx, 'pooled_f1']
ax.axvline(best_thresh, color=C['warn'], linewidth=2, linestyle=':', alpha=0.9)
ax.annotate(f'Best F1 @ τ={best_thresh}\nF1={best_f1:.3f}',
            xy=(best_thresh, best_f1), xytext=(best_thresh + 0.06, best_f1 - 0.05),
            fontsize=10, fontweight='bold', color=C['warn'],
            arrowprops=dict(arrowstyle='->', color=C['warn'], lw=1.5))

ax.set_xlabel('Decision Threshold (τ)', fontweight='bold', fontsize=12)
ax.set_ylabel('Metric Score', fontweight='bold', fontsize=12)
ax.set_title('Decision Threshold Sweep — Logistic Regression',
             fontweight='bold', fontsize=14, pad=12)
ax.legend(fontsize=10, loc='center left', frameon=True,
          facecolor=C['bg'], edgecolor=C['border'])
ax.set_xlim(0.15, 0.65)
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print("\n📊 Threshold Sweep Summary (LR):")
for _, row in thresh_df.iterrows():
    flag = " ← best F1" if row['threshold'] == best_thresh else ""
    print(f"   τ={row['threshold']:.1f}  →  Acc={row['pooled_accuracy']:.3f}  "
          f"Prec={row['pooled_precision']:.3f}  Rec={row['pooled_recall']:.3f}  "
          f"F1={row['pooled_f1']:.3f}{flag}")

---
## §9 — Hyperparameter Sweep Analysis

We swept the TinyKAN hidden dimension and spline range across 18 configurations to find the optimal architecture.

In [ ]:
# ── Sweep Results ────────────────────────────────────────────────────────────
sweep_df = pd.read_csv(OUTPUTS / "openox_tinykan_sweep_results.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── Panel 1: F1 vs Hidden Dim (coloured by spline range) ──
ax = axes[0]
range_colors = {(-2.0, 2.0): C['normal'], (-3.0, 3.0): C['risk'], (-4.0, 4.0): C['accent']}
for (rmin, rmax), color in range_colors.items():
    mask = (sweep_df['spline_range_min'] == rmin) & (sweep_df['spline_range_max'] == rmax)
    subset = sweep_df[mask].sort_values('hidden_dim')
    ax.plot(subset['hidden_dim'], subset['f1'], 'o-', color=color, linewidth=2,
            markersize=8, label=f'Range [{rmin}, {rmax}]',
            markeredgecolor='white', markeredgewidth=1)

# Highlight winner
best_row = sweep_df.iloc[0]
ax.scatter(best_row['hidden_dim'], best_row['f1'], s=200, color=C['warn'],
           marker='*', zorder=10, edgecolors='white', linewidth=1.5)
ax.annotate(f"Best: h={int(best_row['hidden_dim'])}",
            xy=(best_row['hidden_dim'], best_row['f1']),
            xytext=(best_row['hidden_dim']+3, best_row['f1']-0.0005),
            fontsize=9, fontweight='bold', color=C['warn'],
            arrowprops=dict(arrowstyle='->', color=C['warn'], lw=1.2))

ax.set_xlabel('Hidden Dimension', fontweight='bold')
ax.set_ylabel('Pooled F1-Score', fontweight='bold')
ax.set_title('F1 vs. Hidden Dim', fontweight='bold', pad=10)
ax.legend(fontsize=9, loc='lower right', frameon=True, facecolor=C['bg'], edgecolor=C['border'])
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 2: Params vs F1 (Pareto front) ──
ax = axes[1]
scatter = ax.scatter(sweep_df['param_count'], sweep_df['f1'],
                     c=sweep_df['recall'], cmap='RdYlGn', s=100,
                     edgecolors='white', linewidth=1, zorder=3)
cbar = fig.colorbar(scatter, ax=ax, shrink=0.8, label='Recall')
cbar.ax.yaxis.label.set_color(C['text'])
cbar.ax.tick_params(colors=C['dim'])

# Highlight chosen config
chosen = sweep_df[(sweep_df['hidden_dim']==12) & (sweep_df['spline_range_min']==-3.0)].iloc[0]
ax.scatter(chosen['param_count'], chosen['f1'], s=250, facecolors='none',
           edgecolors=C['warn'], linewidth=3, zorder=10)
ax.annotate(f"Chosen\n(297 params)",
            xy=(chosen['param_count'], chosen['f1']),
            xytext=(chosen['param_count']+120, chosen['f1']-0.001),
            fontsize=9, fontweight='bold', color=C['warn'],
            arrowprops=dict(arrowstyle='->', color=C['warn'], lw=1.2))

ax.set_xlabel('Parameter Count', fontweight='bold')
ax.set_ylabel('Pooled F1-Score', fontweight='bold')
ax.set_title('Parameter Efficiency Frontier', fontweight='bold', pad=10)
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 3: Heatmap of F1 ──
ax = axes[2]
pivot = sweep_df.pivot_table(values='f1', index='hidden_dim',
                              columns='spline_range_max', aggfunc='first')
pivot = pivot.sort_index(ascending=False)
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels([f'±{v}' for v in pivot.columns], fontsize=10)
ax.set_yticklabels(pivot.index, fontsize=10)
ax.set_xlabel('Spline Range', fontweight='bold')
ax.set_ylabel('Hidden Dimension', fontweight='bold')
ax.set_title('F1-Score Heatmap', fontweight='bold', pad=10)

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.4f}", ha='center', va='center',
                fontsize=9, fontweight='bold', color='white')
ax.spines[:].set_visible(False)

fig.suptitle("TinyKAN Hyperparameter Sweep — 18 Configurations",
             fontsize=16, fontweight='bold', color=C['text'], y=1.02)
plt.tight_layout()
plt.show()

print("\n🏆 Top 5 Configurations by Pooled F1:")
print(sweep_df.head(5)[['hidden_dim','spline_range_min','spline_range_max',
                         'param_count','best_threshold','f1','recall']].to_string(index=False))

---
## §10 — Global Explainability — Feature Importance

### Logistic Regression: Linear Coefficient Attribution
Each feature's global importance = mean |coefficient| across all 5 folds.

### TinyKAN: Mean-Perturbation Attribution
Attribution$_i = P(\text{baseline}) - P(\text{perturbed with } x_i = \mu_i)$ — averaged over all test windows.

In [ ]:
# ── Global Feature Importance — LR vs TinyKAN ────────────────────────────────
coef_df = pd.read_csv(OUTPUTS / "openox_logreg_global_explainability.csv")
attr_df = pd.read_csv(OUTPUTS / "openox_tinykan_feature_attributions.csv")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ── LR Coefficients ──
coef_sorted = coef_df.sort_values('mean_absolute_coefficient', ascending=True)
labels_lr = [FEATURE_LABELS.get(n, n) for n in coef_sorted['feature_name']]
values_lr = coef_sorted['mean_absolute_coefficient'].values
directions = coef_sorted['dominant_direction'].values
colors_lr = [C['risk'] if d == 'pushes_toward_low_oxygen_risk' else C['normal'] for d in directions]

bars = ax1.barh(range(len(labels_lr)), values_lr, color=colors_lr,
                height=0.65, edgecolor='none', zorder=3)

# Value annotations
for bar, val in zip(bars, values_lr):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", ha='left', va='center', fontsize=9, color=C['text'])

ax1.set_yticks(range(len(labels_lr)))
ax1.set_yticklabels(labels_lr, fontsize=10)
ax1.set_xlabel('|Coefficient| (standardized)', fontweight='bold')
ax1.set_title('Logistic Regression — Global Feature Importance',
              fontweight='bold', pad=12)
ax1.grid(axis='x', alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Legend
legend_patches = [
    mpatches.Patch(color=C['risk'], label='→ Low-O₂ risk'),
    mpatches.Patch(color=C['normal'], label='→ Normal'),
]
ax1.legend(handles=legend_patches, fontsize=9, loc='lower right',
           frameon=True, facecolor=C['bg'], edgecolor=C['border'])

# ── TinyKAN Attributions ──
attr_sorted = attr_df.sort_values('mean_abs_attribution', ascending=True)
labels_tk = [FEATURE_LABELS.get(n, n) for n in attr_sorted['feature_name']]
values_tk = attr_sorted['mean_abs_attribution'].values
signed_tk = attr_sorted['mean_signed_attribution'].values
colors_tk = [C['risk'] if s > 0 else C['normal'] for s in signed_tk]

bars2 = ax2.barh(range(len(labels_tk)), values_tk, color=colors_tk,
                 height=0.65, edgecolor='none', zorder=3)

for bar, val in zip(bars2, values_tk):
    ax2.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", ha='left', va='center', fontsize=9, color=C['text'])

ax2.set_yticks(range(len(labels_tk)))
ax2.set_yticklabels(labels_tk, fontsize=10)
ax2.set_xlabel('Mean |Attribution| (gradient × input)', fontweight='bold')
ax2.set_title('TinyKAN — Feature Attributions (Gradient-Based)',
              fontweight='bold', pad=12)
ax2.grid(axis='x', alpha=0.2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

legend_patches2 = [
    mpatches.Patch(color=C['risk'], label='Positive signed attr.'),
    mpatches.Patch(color=C['normal'], label='Negative signed attr.'),
]
ax2.legend(handles=legend_patches2, fontsize=9, loc='lower right',
           frameon=True, facecolor=C['bg'], edgecolor=C['border'])

fig.suptitle("Global Feature Importance — LR Coefficients vs. TinyKAN Attributions",
             fontsize=16, fontweight='bold', color=C['text'], y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Importance Rank Comparison ───────────────────────────────────────────────
# Are LR and TinyKAN ranking features similarly?

lr_rank = coef_df.sort_values('mean_absolute_coefficient', ascending=False)['feature_name'].reset_index(drop=True)
tk_rank = attr_df.sort_values('mean_abs_attribution', ascending=False)['feature_name'].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 7))

n = len(lr_rank)
for i in range(n):
    feat = lr_rank[i]
    j = tk_rank[tk_rank == feat].index[0]
    
    # Determine color by rank change
    delta = i - j
    if abs(delta) <= 1:
        color = C['accent']  # stable
        alpha = 0.4
    elif abs(delta) <= 3:
        color = C['warn']    # moderate shift
        alpha = 0.5
    else:
        color = C['risk']    # big shift
        alpha = 0.7
    
    ax.plot([0, 1], [n-1-i, n-1-j], '-', color=color, linewidth=2, alpha=alpha)

# Labels
for i in range(n):
    ax.text(-0.05, n-1-i, f"{i+1}. {FEATURE_LABELS.get(lr_rank[i], lr_rank[i])}",
            ha='right', va='center', fontsize=9, color=C['text'], fontweight='bold')
    ax.text(1.05, n-1-i, f"{i+1}. {FEATURE_LABELS.get(tk_rank[i], tk_rank[i])}",
            ha='left', va='center', fontsize=9, color=C['text'], fontweight='bold')

ax.text(0, n + 0.3, 'LR Rank', ha='center', fontsize=12, fontweight='bold', color=C['normal'])
ax.text(1, n + 0.3, 'TinyKAN Rank', ha='center', fontsize=12, fontweight='bold', color=C['risk'])

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-1, n + 1)
ax.axis('off')
ax.set_title('Feature Importance Rank Comparison — LR vs TinyKAN',
             fontweight='bold', fontsize=14, pad=20, color=C['text'])

# Legend
legend_elements = [
    plt.Line2D([0], [0], color=C['accent'], linewidth=2, label='Stable (Δ ≤ 1)'),
    plt.Line2D([0], [0], color=C['warn'], linewidth=2, label='Moderate shift (Δ 2–3)'),
    plt.Line2D([0], [0], color=C['risk'], linewidth=2, label='Large shift (Δ > 3)'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower center',
          frameon=True, facecolor=C['bg'], edgecolor=C['border'])

plt.tight_layout()
plt.show()

---
## §11 — Local Explainability — Per-Window Attribution

How does the model explain its prediction for a **single window**?

- **LR**: Contribution$_i = w_i \times x_{i,\text{scaled}}$ — a signed bar chart showing each feature's push.
- **TinyKAN**: Mean-Perturbation — zeroing feature $i$ to its training mean and measuring probability drop.

In [ ]:
# ── Local Explanation Waterfall (LR Baseline) ────────────────────────────────
# Reconstruct a single fold's model and explain one positive window

scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_all)

model_full = LogisticRegression(class_weight='balanced', max_iter=1000,
                                 solver='liblinear', random_state=42)
model_full.fit(X_full_scaled, y_all)

coef = model_full.coef_[0]
intercept = model_full.intercept_[0]

# Find a confident positive window
probs_full = model_full.predict_proba(X_full_scaled)[:, 1]
pos_mask = (y_all == 1) & (probs_full > 0.85)
example_idx = np.where(pos_mask)[0][0]

x_example = X_full_scaled[example_idx]
contributions = x_example * coef
prob_example = probs_full[example_idx]

# Waterfall chart — top 10 features
order = np.argsort(np.abs(contributions))[::-1][:10][::-1]
wf_labels = [FEATURE_LABELS.get(FEATURE_NAMES[i], FEATURE_NAMES[i]) for i in order]
wf_vals = contributions[order]
wf_colors = [C['risk'] if v > 0 else C['normal'] for v in wf_vals]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                gridspec_kw={'width_ratios': [2.5, 1]})

# ── Waterfall ──
bars = ax1.barh(range(len(wf_labels)), wf_vals, color=wf_colors,
                height=0.6, edgecolor='none', zorder=3)
ax1.axvline(0, color=C['dim'], linewidth=1, zorder=2)
ax1.set_yticks(range(len(wf_labels)))
ax1.set_yticklabels(wf_labels, fontsize=10)
ax1.set_xlabel('Feature Contribution (w × x_scaled)', fontweight='bold')
ax1.set_title(f'Local Explanation — Window #{example_idx} (P(risk)={prob_example:.2f})',
              fontweight='bold', pad=12)

# Annotation
ax1.text(0.02, 0.96, '→ Toward low-oxygen risk', transform=ax1.transAxes,
         fontsize=9, color=C['risk'], fontweight='bold', va='top')
ax1.text(0.02, 0.90, '← Toward normal', transform=ax1.transAxes,
         fontsize=9, color=C['normal'], fontweight='bold', va='top')

for bar, val in zip(bars, wf_vals):
    ha = 'left' if val > 0 else 'right'
    offset = 0.02 if val > 0 else -0.02
    ax1.text(val + offset, bar.get_y() + bar.get_height()/2,
             f"{val:+.3f}", ha=ha, va='center', fontsize=8, color=C['text'])

ax1.grid(axis='x', alpha=0.15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# ── Cumulative contribution breakdown ──
sorted_abs_order = np.argsort(np.abs(contributions))[::-1]
cumulative = np.cumsum(np.abs(contributions[sorted_abs_order]))
cumulative_pct = cumulative / cumulative[-1] * 100

ax2.fill_between(range(1, 15), cumulative_pct, alpha=0.3, color=C['accent'])
ax2.plot(range(1, 15), cumulative_pct, 'o-', color=C['accent'], linewidth=2,
         markersize=6, markeredgecolor='white', markeredgewidth=1)
ax2.axhline(90, color=C['warn'], linewidth=1.5, linestyle='--', alpha=0.7, label='90% threshold')

# Find where 90% is reached
n_90 = np.searchsorted(cumulative_pct, 90) + 1
ax2.axvline(n_90, color=C['warn'], linewidth=1, linestyle=':', alpha=0.5)
ax2.annotate(f'{n_90} features\n= 90%',
             xy=(n_90, 90), xytext=(n_90+2, 75),
             fontsize=9, fontweight='bold', color=C['warn'],
             arrowprops=dict(arrowstyle='->', color=C['warn'], lw=1.2))

ax2.set_xlabel('Number of Features', fontweight='bold')
ax2.set_ylabel('Cumulative Contribution (%)', fontweight='bold')
ax2.set_title('Explanation Compactness', fontweight='bold', pad=12)
ax2.set_xlim(0.5, 14.5)
ax2.set_ylim(0, 105)
ax2.legend(fontsize=9, loc='lower right')
ax2.grid(alpha=0.2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Multi-Window Local Explanation Grid ──────────────────────────────────────
# Show explanations for 4 representative windows (TP, FP, TN, FN)

preds_full = (probs_full >= 0.5).astype(int)

quadrants = {
    'True Positive': (y_all == 1) & (preds_full == 1),
    'False Positive': (y_all == 0) & (preds_full == 1),
    'True Negative': (y_all == 0) & (preds_full == 0),
    'False Negative': (y_all == 1) & (preds_full == 0),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
quadrant_colors = {
    'True Positive': C['accent'],
    'False Positive': C['warn'],
    'True Negative': C['normal'],
    'False Negative': C['risk'],
}

for (name, mask), ax in zip(quadrants.items(), axes.ravel()):
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        ax.text(0.5, 0.5, f'{name}\nNo samples', ha='center', va='center',
                fontsize=12, color=C['dim'])
        ax.axis('off')
        continue
    
    # Pick a mid-confidence example
    idx = idxs[len(idxs)//2]
    x_ex = X_full_scaled[idx]
    contribs = x_ex * coef
    prob_ex = probs_full[idx]
    
    top8 = np.argsort(np.abs(contribs))[::-1][:8][::-1]
    lbl = [FEATURE_LABELS.get(FEATURE_NAMES[k], FEATURE_NAMES[k]) for k in top8]
    vals = contribs[top8]
    clrs = [C['risk'] if v > 0 else C['normal'] for v in vals]
    
    ax.barh(range(len(lbl)), vals, color=clrs, height=0.6, edgecolor='none', zorder=3)
    ax.axvline(0, color=C['dim'], linewidth=0.8)
    ax.set_yticks(range(len(lbl)))
    ax.set_yticklabels(lbl, fontsize=9)
    
    # Title with colored badge
    ax.set_title(f"{name}  ·  P(risk) = {prob_ex:.3f}  ·  True = {y_all[idx]}",
                 fontweight='bold', pad=8, fontsize=11,
                 color=quadrant_colors[name])
    
    ax.grid(axis='x', alpha=0.15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle("Local Explanations Across Confusion Matrix Quadrants",
             fontsize=16, fontweight='bold', color=C['text'], y=1.01)
plt.tight_layout()
plt.show()

---
## §12 — Explanation Quality Rubric Scoring

Each explanation is scored against a **3-criterion rubric** (max 6 points):

| Criterion | Max | Description |
| :--- | :---: | :--- |
| **Compactness** | 2 | ≤ 30 words |
| **Consistency** | 2 | Top feature direction matches prediction |
| **Usefulness** | 2 | Includes probability, dominant feature, and direction |

**Pass gate**: mean total ≥ 4.0, no zeroes in any criterion.

In [ ]:
# ── Rubric Scoring Results ───────────────────────────────────────────────────
rubric_summary = pd.read_csv(OUTPUTS / "openox_logreg_explanation_rubric_summary.csv")
rubric_detail = pd.read_csv(OUTPUTS / "openox_logreg_explanation_sample.csv")

rs = rubric_summary.iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Rubric Score Breakdown ──
ax = axes[0]
criteria = ['Compactness', 'Consistency', 'Usefulness']
means = [rs['mean_compactness'], rs['mean_consistency'], rs['mean_usefulness']]
max_vals = [2, 2, 2]
colors_rubric = [C['cyan'], C['purple'], C['accent']]

bars = ax.barh(criteria[::-1], means[::-1], color=colors_rubric[::-1],
               height=0.5, edgecolor='none', zorder=3)
# Max reference
ax.barh(criteria[::-1], max_vals[::-1], color=C['border'], height=0.5,
        edgecolor='none', alpha=0.3, zorder=2)

for bar, val, mx in zip(bars, means[::-1], max_vals[::-1]):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}/{mx}", ha='left', va='center',
            fontsize=11, fontweight='bold', color=C['text'])

ax.set_xlim(0, 2.5)
ax.set_title('Mean Rubric Scores (n=20)', fontweight='bold', pad=12)
ax.grid(axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Panel 2: Overall Score Gauge ──
ax = axes[1]
total_score = rs['mean_total_score']
passed = rs['passes_research_plan_gate']

# Donut gauge
fill = total_score / 6.0
remaining = 1 - fill
gauge_colors = [C['accent'] if passed else C['risk'], C['border']]

wedges, _ = ax.pie([fill, remaining], colors=gauge_colors,
                    startangle=90, wedgeprops=dict(width=0.35, edgecolor=C['bg'], linewidth=2))
centre = plt.Circle((0,0), 0.6, fc=C['bg'])
ax.add_artist(centre)
ax.text(0, 0.08, f"{total_score:.1f}", ha='center', va='center',
        fontsize=36, fontweight='bold', color=C['accent'] if passed else C['risk'])
ax.text(0, -0.18, 'out of 6.0', ha='center', va='center',
        fontsize=11, color=C['dim'])
ax.text(0, -0.42, '✅ PASS' if passed else '❌ FAIL', ha='center', va='center',
        fontsize=14, fontweight='bold',
        color=C['accent'] if passed else C['risk'])
ax.set_title('Rubric Gate Assessment', fontweight='bold', pad=20)

# ── Panel 3: Per-Quadrant Scores ──
ax = axes[2]
if 'quadrant' in rubric_detail.columns and 'total' in rubric_detail.columns:
    quadrant_means = rubric_detail.groupby('quadrant')['total'].mean()
    q_order = ['TP', 'FP', 'TN', 'FN']
    q_colors = [C['accent'], C['warn'], C['normal'], C['risk']]
    q_vals = [quadrant_means.get(q, 0) for q in q_order]
    
    bars = ax.bar(q_order, q_vals, color=q_colors, width=0.55,
                  edgecolor='none', zorder=3)
    ax.axhline(4.0, color=C['warn'], linewidth=2, linestyle='--', alpha=0.7, label='Pass gate (≥ 4.0)')
    
    for bar, val in zip(bars, q_vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
                f"{val:.1f}", ha='center', fontsize=11, fontweight='bold', color=C['text'])
    
    ax.set_ylim(0, 7)
    ax.set_ylabel('Mean Total Score', fontweight='bold')
    ax.set_title('Score by CM Quadrant', fontweight='bold', pad=12)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(axis='y', alpha=0.2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle("Explanation Quality Assessment — 3-Criterion Rubric",
             fontsize=16, fontweight='bold', color=C['text'], y=1.02)
plt.tight_layout()
plt.show()

print(f"\n📋 Rubric Summary:")
print(f"   Mean Compactness:  {rs['mean_compactness']:.2f}/2.00")
print(f"   Mean Consistency:  {rs['mean_consistency']:.2f}/2.00")
print(f"   Mean Usefulness:   {rs['mean_usefulness']:.2f}/2.00")
print(f"   Mean Total:        {total_score:.2f}/6.00")
print(f"   Gate Status:       {'✅ PASS' if passed else '❌ FAIL'}")

In [ ]:
# ── Sample Explanation Table ──────────────────────────────────────────────────
if 'explanation' in rubric_detail.columns:
    display_cols = ['window_id', 'quadrant', 'probability', 'top1_feature',
                    'compactness', 'consistency', 'usefulness', 'total']
    existing_cols = [c for c in display_cols if c in rubric_detail.columns]
    display_df = rubric_detail[existing_cols].head(10)
    print("\n📝 Sample Explanation Rubric Scores (first 10):")
    print(display_df.to_string(index=False))

---
## §13 — TinyML Deployment Summary

The trained models are deployed on an **Arduino Nano 33 BLE** microcontroller:

| Property | Logistic Regression | TinyKAN |
| :--- | :---: | :---: |
| Parameters | 15 | 297 |
| Flash footprint | ~120 bytes | ~2.4 KB |
| XAI method | Linear coefficients | Mean-Perturbation (15 passes) |
| Output | Risk flag + Top-3 drivers | Risk flag + Top-3 drivers |
| Wireless | BLE → Web Bluetooth dashboard | BLE → Web Bluetooth dashboard |

In [ ]:
# ── System Architecture Diagram (text-based) ────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor(C['bg'])

# ── Pipeline blocks ──
blocks = [
    (1.0, 7.0, 2.5, 1.2, 'MAX30105\nOptical Sensor', C['normal'], '50Hz Raw PPG'),
    (4.5, 7.0, 2.5, 1.2, 'Bandpass Filter\n0.5–8 Hz', C['purple'], 'Filtered Signal'),
    (8.0, 7.0, 2.5, 1.2, 'Sliding Window\n10s / 500 samples', C['cyan'], '14 Features'),
    (11.5, 7.0, 3.0, 1.2, 'Feature Extraction\n14 PPG Features', C['accent'], ''),
    
    (2.0, 4.2, 3.0, 1.5, 'Logistic Regression\n15 params\nLinear Coefficients', C['normal'], ''),
    (8.0, 4.2, 3.0, 1.5, 'TinyKAN (14→12→1)\n297 params\nMean-Perturbation', C['risk'], ''),
    
    (5.0, 1.5, 4.0, 1.2, 'Risk Flag + Top-3 Drivers\nBLE Telemetry', C['warn'], ''),
    (10.5, 1.5, 4.0, 1.2, 'Web Bluetooth\nDashboard', C['accent'], ''),
]

for x, y, w, h, label, color, arrow_label in blocks:
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                    facecolor=color, alpha=0.2,
                                    edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color=C['text'])

# ── Arrows ──
arrow_props = dict(arrowstyle='->', color=C['dim'], lw=2, connectionstyle='arc3,rad=0')

# Top row connections
ax.annotate('', xy=(4.5, 7.6), xytext=(3.5, 7.6), arrowprops=arrow_props)
ax.annotate('', xy=(8.0, 7.6), xytext=(7.0, 7.6), arrowprops=arrow_props)
ax.annotate('', xy=(11.5, 7.6), xytext=(10.5, 7.6), arrowprops=arrow_props)

# Down to models
ax.annotate('', xy=(3.5, 5.7), xytext=(12.5, 7.0), arrowprops=dict(
    arrowstyle='->', color=C['dim'], lw=1.5, connectionstyle='arc3,rad=-0.3'))
ax.annotate('', xy=(9.5, 5.7), xytext=(13.0, 7.0), arrowprops=dict(
    arrowstyle='->', color=C['dim'], lw=1.5, connectionstyle='arc3,rad=0.3'))

# Down to output
ax.annotate('', xy=(6.0, 2.7), xytext=(3.5, 4.2), arrowprops=dict(
    arrowstyle='->', color=C['dim'], lw=1.5, connectionstyle='arc3,rad=0.2'))
ax.annotate('', xy=(8.0, 2.7), xytext=(9.5, 4.2), arrowprops=dict(
    arrowstyle='->', color=C['dim'], lw=1.5, connectionstyle='arc3,rad=-0.2'))

# BLE to Dashboard
ax.annotate('', xy=(10.5, 2.1), xytext=(9.0, 2.1), arrowprops=dict(
    arrowstyle='->', color=C['warn'], lw=2.5, connectionstyle='arc3,rad=0'))
ax.text(9.75, 2.5, 'BLE', ha='center', fontsize=8, color=C['warn'], fontweight='bold')

# MCU box
mcu_rect = mpatches.FancyBboxPatch((1.0, 3.5, ), 11.5, 3.5,
                                     boxstyle='round,pad=0.3',
                                     facecolor='none',
                                     edgecolor=C['border'],
                                     linewidth=2, linestyle='--')
ax.add_patch(mcu_rect)
ax.text(6.75, 3.6, 'Arduino Nano 33 BLE — On-Device Processing',
        ha='center', fontsize=10, fontweight='bold', color=C['dim'], style='italic')

ax.set_title('End-to-End TinyML Pipeline — Sensor to Dashboard',
             fontsize=16, fontweight='bold', pad=20, color=C['text'])

plt.tight_layout()
plt.show()

In [ ]:
# ── Final Summary Dashboard ──────────────────────────────────────────────────

fig = plt.figure(figsize=(18, 8))
gs = GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.4)

# ── KPI Cards ──
kpis = [
    ('52,090', 'Total Windows', C['normal']),
    ('162', 'Subjects', C['purple']),
    ('14', 'PPG Features', C['cyan']),
    ('297', 'TinyKAN Params', C['risk']),
]

for i, (value, label, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    rect = mpatches.FancyBboxPatch((0.05, 0.05), 0.9, 0.9,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.12,
                                    edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(0.5, 0.6, value, ha='center', va='center',
            fontsize=28, fontweight='bold', color=color)
    ax.text(0.5, 0.28, label, ha='center', va='center',
            fontsize=11, color=C['dim'])

# ── Bottom panels: LR vs TinyKAN mini comparison ──
ax = fig.add_subplot(gs[1, 0:2])
comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'LR': [lr_primary['pooled_accuracy'], lr_primary['pooled_precision'],
           lr_primary['pooled_recall'], lr_primary['pooled_f1']],
    'TinyKAN': [tk['pooled_accuracy'], tk['pooled_precision'],
               tk['pooled_recall'], tk['pooled_f1']],
}
cmp_df = pd.DataFrame(comparison_data)

x = np.arange(4)
w = 0.3
ax.bar(x - w/2, cmp_df['LR'], w, color=C['normal'], label='LR (15 params)',
       edgecolor='none', zorder=3)
ax.bar(x + w/2, cmp_df['TinyKAN'], w, color=C['risk'], label='TinyKAN (297 params)',
       edgecolor='none', zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(cmp_df['Metric'])
ax.set_ylim(0, 1.1)
ax.set_title('Model Performance', fontweight='bold', pad=8)
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Rubric gauge ──
ax = fig.add_subplot(gs[1, 2])
fill = total_score / 6.0
wedges, _ = ax.pie([fill, 1-fill],
                    colors=[C['accent'], C['border']],
                    startangle=90,
                    wedgeprops=dict(width=0.35, edgecolor=C['bg'], linewidth=2))
centre = plt.Circle((0,0), 0.55, fc=C['bg'])
ax.add_artist(centre)
ax.text(0, 0.05, f"{total_score:.1f}/6", ha='center', va='center',
        fontsize=20, fontweight='bold', color=C['accent'])
ax.text(0, -0.22, 'XAI Rubric', ha='center', fontsize=10, color=C['dim'])
ax.set_title('Explanation Quality', fontweight='bold', pad=15)

# ── Semester 2 Roadmap Checklist ──
ax = fig.add_subplot(gs[1, 3])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

roadmap = (
    "  SEMESTER 2 ROADMAP\n"
    "  ━━━━━━━━━━━━━━━━━━━━━\n\n"
    "  ☐ On-device verification\n"
    "  ☐ Embedded benchmarking\n"
    "  ☐ Stress testing\n"
    "     • Motion artifacts\n"
    "     • Skin tone params\n"
    "     • Ambient light\n"
    "  ☐ User evaluation\n"
    "  ☐ Rubric gate ≥ 4.0"
)
ax.text(0.5, 0.5, roadmap, ha='center', va='center', fontsize=9,
        fontfamily='monospace', color=C['text'],
        bbox=dict(boxstyle='round,pad=0.5', facecolor=C['bg'],
                  edgecolor=C['border'], linewidth=1.5))

fig.suptitle("📊 Project Dashboard — Explainable TinyML for PPG Oxygen Risk Monitoring",
             fontsize=17, fontweight='bold', color=C['text'], y=1.02)
plt.show()

In [ ]:
# ── Simulated PPG Waveform Visualization ─────────────────────────────────────
# Generate a realistic-looking PPG waveform for illustration purposes

np.random.seed(42)
t = np.linspace(0, 10, 500)  # 10s at 50Hz
hr = 1.2  # Hz (72 BPM)

# Simulated PPG: systolic peak + dicrotic notch
phase = 2 * np.pi * hr * t
systolic = np.exp(-((np.mod(phase, 2*np.pi) - 1.0)**2) / 0.08)
dicrotic = 0.3 * np.exp(-((np.mod(phase, 2*np.pi) - 2.5)**2) / 0.15)
baseline = 0.05 * np.sin(2 * np.pi * 0.2 * t)  # respiration
noise = 0.02 * np.random.randn(500)
ppg = systolic + dicrotic + baseline + noise + 0.5

# Bandpass filter
sos = signal.butter(4, [0.5, 8.0], btype='bandpass', fs=50, output='sos')
ppg_filtered = signal.sosfiltfilt(sos, ppg)

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

# ── Raw PPG ──
ax = axes[0]
ax.plot(t, ppg, color=C['risk'], linewidth=1.2, alpha=0.9)
ax.fill_between(t, ppg, alpha=0.08, color=C['risk'])
ax.set_ylabel('Amplitude', fontweight='bold')
ax.set_title('Raw PPG Signal (Simulated — 50 Hz, 72 BPM)', fontweight='bold', pad=8)
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Filtered PPG ──
ax = axes[1]
ax.plot(t, ppg_filtered, color=C['normal'], linewidth=1.5, alpha=0.9)
ax.fill_between(t, ppg_filtered, alpha=0.08, color=C['normal'])
ax.set_ylabel('Amplitude', fontweight='bold')
ax.set_title('Bandpass Filtered (0.5–8 Hz)', fontweight='bold', pad=8)
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Power Spectral Density ──
ax = axes[2]
freqs, psd = signal.welch(ppg_filtered, fs=50, nperseg=256)
ax.semilogy(freqs, psd, color=C['accent'], linewidth=2)
ax.fill_between(freqs, psd, alpha=0.15, color=C['accent'])

# Highlight bandpower regions
mask_low = (freqs >= 0.5) & (freqs <= 2.0)
mask_high = (freqs >= 2.0) & (freqs <= 5.0)
ax.fill_between(freqs[mask_low], psd[mask_low], alpha=0.25, color=C['warn'],
                label='0.5–2 Hz (respiration)')
ax.fill_between(freqs[mask_high], psd[mask_high], alpha=0.25, color=C['purple'],
                label='2–5 Hz (cardiac)')

# Mark dominant frequency
dom_idx = np.argmax(psd[1:]) + 1
ax.axvline(freqs[dom_idx], color=C['risk'], linewidth=1.5, linestyle='--', alpha=0.7)
ax.annotate(f'Dominant: {freqs[dom_idx]:.2f} Hz',
            xy=(freqs[dom_idx], psd[dom_idx]),
            xytext=(freqs[dom_idx]+2, psd[dom_idx]*0.5),
            fontsize=9, fontweight='bold', color=C['risk'],
            arrowprops=dict(arrowstyle='->', color=C['risk'], lw=1.2))

ax.set_xlabel('Frequency (Hz)', fontweight='bold')
ax.set_ylabel('PSD', fontweight='bold')
ax.set_title('Power Spectral Density (Welch)', fontweight='bold', pad=8)
ax.set_xlim(0, 10)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle('PPG Signal Processing Pipeline — From Raw Waveform to Spectral Features',
             fontsize=16, fontweight='bold', color=C['text'], y=1.02)
plt.tight_layout()
plt.show()

---

## 🏁 Notebook Complete

This notebook has walked through the **complete ML workflow** for the Explainable TinyML PPG Oxygen Risk Monitoring project:

1. ✅ **Data**: 52,090 windows from 162 subjects (OpenOximetry)
2. ✅ **Features**: 14 physiological PPG features across 4 categories
3. ✅ **Baseline**: Logistic Regression — F1 = 0.726, Recall = 86.6%
4. ✅ **Candidate**: TinyKAN (14→12→1) — F1 = 0.744, Recall = 97.1%
5. ✅ **Explainability**: LR coefficients + TinyKAN mean-perturbation
6. ✅ **Rubric**: Perfect 6.0/6.0 — all criteria passed
7. ✅ **Deployment**: Arduino Nano 33 BLE with BLE telemetry

### Next Steps (Semester 2)
- On-device verification (C++ ↔ Python agreement)
- Embedded resource benchmarking (Flash, RAM, latency)
- Environmental stress testing (motion, skin tone, ambient light)
- User evaluation & live rubric validation